# Job Market Skill Analyzer
**Assignment 01 — Python Data Structures**

This notebook builds a compact tool to store job posting data, add new postings, clean and analyze requested skills, and export structured output files.

## Setup: Job Dataset

In [ ]:
job_data = {
    "Job_ID": [1, 2, 3, 4, 5],
    "Role": ["Data Analyst", "Data Scientist", "Business Analyst", "Data Engineer", "ML Engineer"],
    "Company": ["ABC Corp", "XyZ Ltd", "DataWorks", "InfraTech", "AI Labs"],
    "Skills": [
        "Excel SQL Python",
        "Python ML Statistics",
        "Excel SQL PowerBI",
        "Python SQL Spark",
        "Python ML DeepLearning"
    ]
}
print(job_data)

## Task 1: Print Each Job Posting
Loop through `job_data` and print each posting in a readable format.

In [ ]:
for i in range(len(job_data["Job_ID"])):
    print(f"ID: {job_data['Job_ID'][i]} | Role: {job_data['Role'][i]} | "
          f"Company: {job_data['Company'][i]} | Skills: {job_data['Skills'][i]}")

## Task 2: Add New Job Postings (Interactive)
Ask the user how many postings to add, collect role/company/skills for each, auto-increment `Job_ID` from the current max, and append to `job_data`.

In [ ]:
num_new = int(input("How many job postings do you want to add? "))

for _ in range(num_new):
    new_role = input("Enter role: ").strip()
    new_company = input("Enter company: ").strip()
    new_skills = input("Enter skills (space-separated or comma-separated): ").strip()

    # Auto-increment Job_ID from the current max, cast to int
    new_id = int(max(job_data["Job_ID"]) + 1)

    job_data["Job_ID"].append(new_id)
    job_data["Role"].append(new_role)
    job_data["Company"].append(new_company)
    job_data["Skills"].append(new_skills)

print("\nUpdated job_data:")
print(job_data)

## Task 3: Clean & Normalize Skills
`clean_skills()` lowercases text, replaces commas with spaces, strips punctuation, and normalizes whitespace. Apply it to every entry in `job_data['Skills']`.

In [ ]:
import re

def clean_skills(skills_str):
    """
    Cleans a raw skills string:
    - lowercases text
    - replaces commas with spaces
    - removes punctuation
    - normalizes whitespace
    """
    text = skills_str.lower()
    text = text.replace(",", " ")
    text = re.sub(r"[^\w\s]", " ", text)     # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()  # normalize whitespace
    return text

job_data["Skills"] = [clean_skills(s) for s in job_data["Skills"]]
print(job_data["Skills"])

## Task 4: Skill Extraction & Frequency
Split cleaned skill strings into tokens, build a consolidated skill list, compute frequency counts, and print the top 5 most frequent skills.

In [ ]:
all_skills = []
for skill_str in job_data["Skills"]:
    tokens = skill_str.split()
    all_skills.extend(tokens)

skill_count = {}
for skill in all_skills:
    skill_count[skill] = skill_count.get(skill, 0) + 1

sorted_skills = sorted(skill_count.items(), key=lambda x: x[1], reverse=True)

print("Top skills by frequency:")
for skill, count in sorted_skills[:5]:
    print(f"{skill}: {count}")

## Task 5: Basic Statistics & Insights

In [ ]:
total_jobs = len(job_data["Job_ID"])
unique_skills_count = len(set(all_skills))
python_jobs = [job_data["Role"][i] for i in range(len(job_data["Job_ID"]))
               if "python" in job_data["Skills"][i].lower()]

print(f"Total Jobs: {total_jobs}")
print(f"Unique skills: {unique_skills_count}")
print(f"Jobs mentioning Python: {python_jobs}")

## Task 6: Skill Search Function
`find_jobs_by_skill()` normalizes the query (lowercase, strip) and returns matching `(Job_ID, Role, Company)` tuples.

In [ ]:
def find_jobs_by_skill(skill_query):
    """
    Returns a list of (Job_ID, Role, Company) tuples for postings
    that contain the given skill_query token.
    """
    query = skill_query.lower().strip()
    results = []

    if not query:
        return results

    for i in range(len(job_data["Job_ID"])):
        tokens = job_data["Skills"][i].split()
        if query in tokens:
            results.append((job_data["Job_ID"][i], job_data["Role"][i], job_data["Company"][i]))
        elif query in job_data["Skills"][i]:
            # catches partial/substring matches too (e.g. "power" in "powerbi")
            results.append((job_data["Job_ID"][i], job_data["Role"][i], job_data["Company"][i]))
        else:
            continue

    return results

# Example usage
print(find_jobs_by_skill("sql"))

## Task 7: Lambda Filtering & List Comprehension
Use `filter()` with a lambda to find roles that include at least one "common" skill (frequency >= 2).

In [ ]:
common_skill_indices = list(filter(
    lambda i: any(skill_count.get(tok, 0) >= 2 for tok in job_data["Skills"][i].split()),
    range(len(job_data["Job_ID"]))
))
roles_with_common_skill = [job_data["Role"][i] for i in common_skill_indices]

print("Roles with at least one common skill:", roles_with_common_skill)

## Task 8: Unique Skill Set

In [ ]:
unique_skills = sorted(set(all_skills))
print("Unique skills:")
print(unique_skills)

## Task 9: Save Cleaned Data to File
Write the cleaned records to `job_skills.csv` and the top skills to `top_skills.txt`, using `try/except/else/finally` for error handling.

In [ ]:
try:
    with open("job_skills.csv", "w", newline="") as f:
        f.write("Job_ID,Role,Company,Skills\n")
        for i in range(len(job_data["Job_ID"])):
            f.write(f"{job_data['Job_ID'][i]},{job_data['Role'][i]},"
                     f"{job_data['Company'][i]},{job_data['Skills'][i]}\n")
except IOError as e:
    print(f"Error writing job_skills.csv: {e}")
else:
    print("job_skills.csv written successfully.")
finally:
    print("Finished attempt to write job_skills.csv.")

In [ ]:
try:
    with open("top_skills.txt", "w") as f:
        f.write("Top Skills by Frequency\n")
        f.write("========================\n")
        for skill, count in sorted_skills:
            f.write(f"{skill}: {count}\n")
except IOError as e:
    print(f"Error writing top_skills.txt: {e}")
else:
    print("top_skills.txt written successfully.")
finally:
    print("Finished attempt to write top_skills.txt.")

---
### Notes
- Run all cells top to bottom. Task 2's cell will prompt for input in Colab/Jupyter — enter the number of postings and their details when asked.
- Running this notebook produces `job_skills.csv` and `top_skills.txt` in the same directory as the notebook — upload both to the GitHub repo alongside the `.ipynb` file.